In [1]:
import torch
from PIL import Image
import requests
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [2]:
# 1. Load sample image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

In [3]:
# 2. Load ConvNeXt processor and model
checkpoint = "facebook/convnext-tiny-224"
processor = AutoImageProcessor.from_pretrained(checkpoint)
model = AutoModelForImageClassification.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  114MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

In [ ]:
# 3. Preprocess and infer
inputs = processor(images=image, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print("ConvNeXt Prediction:", model.config.id2label[predicted_class_idx])

ConvNeXt Prediction: tabby, tabby cat


In [4]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 15.0 MB/s eta 0:00:00


In [14]:
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torch.nn as nn
import medmnist
from medmnist import INFO

In [6]:
# 2. Dataset Configuration
DATASET_FLAG = 'pneumoniamnist'  # 2D Binary Image Dataset (0: Normal, 1: Pneumonia)
info = INFO[DATASET_FLAG]
num_classes = len(info['label'])  # 2 classes
DataClass = getattr(medmnist, info['python_class'])

In [21]:
CONVNEXT_CHECKPOINT = "facebook/convnext-tiny-224"
SWIN_CHECKPOINT = "microsoft/swin-tiny-patch4-window7-224"

In [8]:
# Convert 1-channel grayscale X-ray to 3-channel 224x224 RGB image for HF backbone
data_transform = T.Compose([
    T.Resize((224, 224)),
    T.Lambda(lambda image: image.convert("RGB")),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [37]:
# Load datasets and dataloaders
train_dataset = DataClass(split='train', transform=data_transform, download=True)
val_dataset = DataClass(split='val', transform=data_transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [23]:
# Load Pretrained Base Model
cvnxt_model = AutoModelForImageClassification.from_pretrained(
    CONVNEXT_CHECKPOINT,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

swin_model = AutoModelForImageClassification.from_pretrained(
    SWIN_CHECKPOINT,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

[transformers] ConvNextForImageClassification LOAD REPORT from: facebook/convnext-tiny-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [26]:
#  Apply Feature Extraction Pattern (Freeze Backbone)
for name, param in cvnxt_model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

#  Apply Feature Extraction Pattern (Freeze Backbone)
for name, param in swin_model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

In [27]:
# Optimizer & Criterion
cvnxt_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.01
)

swin_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=0.01
)

criterion = nn.CrossEntropyLoss()

In [32]:
# Verify setup
cvnxt_trainable_params = sum(p.numel() for p in cvnxt_model.parameters() if p.requires_grad)
cvnxt_total_params = sum(p.numel() for p in cvnxt_model.parameters())
print(f"Dataset Loaded: {DATASET_FLAG} ({num_classes} classes)")
print(f"Feature Extraction Setup Complete | ConvNext Trainable Params: {cvnxt_trainable_params:,} / {cvnxt_total_params:,}")

# Verify setup
swin_trainable_params = sum(p.numel() for p in swin_model.parameters() if p.requires_grad)
swin_total_params = sum(p.numel() for p in swin_model.parameters())
print(f"\nDataset Loaded: {DATASET_FLAG} ({num_classes} classes)")
print(f"Feature Extraction Setup Complete | Swin Trainable Params: {swin_trainable_params:,} / {swin_total_params:,}")

Dataset Loaded: pneumoniamnist (2 classes)
Feature Extraction Setup Complete | ConvNext Trainable Params: 1,538 / 27,821,666

Dataset Loaded: pneumoniamnist (2 classes)
Feature Extraction Setup Complete | Swin Trainable Params: 1,538 / 27,520,892


In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

cvnxt_model.to(device)

cvnxt_model.train()
cvnxt_running_loss = 0.0

for images, labels in train_loader:
    images = images.to(device)
    # MedMNIST outputs 2D targets [[0], [1]]; squeeze to 1D tensor [0, 1] for CrossEntropyLoss
    labels = labels.squeeze(1).long().to(device)
    
    cvnxt_optimizer.zero_grad()
    outputs = cvnxt_model(images)
    loss = criterion(outputs.logits, labels)
    loss.backward()
    cvnxt_optimizer.step()
    
    cvnxt_running_loss += loss.item() * images.size(0)

epoch_loss = cvnxt_running_loss / len(train_loader.dataset)
print(f"Training Epoch Loss: {epoch_loss:.4f}")

Training Epoch Loss: 0.6612


In [ ]:

swin_model.to(device)

swin_model.train()
swin_running_loss = 0.0

for images, labels in train_loader:
    images = images.to(device)
    # MedMNIST outputs 2D targets [[0], [1]]; squeeze to 1D tensor [0, 1] for CrossEntropyLoss
    labels = labels.squeeze(1).long().to(device)
    
    swin_optimizer.zero_grad()
    outputs = swin_model(images)
    loss = criterion(outputs.logits, labels)
    loss.backward()
    swin_optimizer.step()
    
    swin_running_loss += loss.item() * images.size(0)

epoch_loss = swin_running_loss / len(train_loader.dataset)
print(f"Training Epoch Loss: {epoch_loss:.4f}")

Training Epoch Loss: 0.6106
